In [ ]:
!pip install gradio ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.3 MB/s eta 0:00:00


In [ ]:
import os, cv2, warnings
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
import timm
from ultralytics import YOLO
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches

warnings.filterwarnings("ignore")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
EFFNET_CKPT = "/content/drive/MyDrive/models/best_effnet_v4.pth"
YOLO_CKPT   = "/content/drive/MyDrive/models/best.pt"

ENSEMBLE_CONFIG = {
    "effnet_threshold" : 0.6623,   # Youden optimal from v4
    "yolo_conf"        : 0.15,     # best recall threshold from sweep
    "yolo_iou"         : 0.45,
    "yolo_imgsz"       : 640,
    "effnet_weight"    : 0.45,
    "yolo_weight"      : 0.55,
    "final_threshold"  : 0.4355,
    "use_clahe"        : True,
    "border_crop_pct"  : 0.04,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [ ]:
def remove_xray_border(pil_img, pct=ENSEMBLE_CONFIG["border_crop_pct"]):
    img    = np.array(pil_img)
    h, w   = img.shape[:2]
    mh, mw = int(h * pct), int(w * pct)
    return Image.fromarray(img[mh:h-mh, mw:w-mw])

def apply_clahe(pil_img):
    img_np   = np.array(pil_img.convert("L"))
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(img_np)
    return remove_xray_border(Image.fromarray(enhanced))

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

infer_transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
def load_effnet(ckpt_path):
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg   = ckpt["config"]
    m     = timm.create_model(
        cfg["model_name"], pretrained=False,
        num_classes=cfg["num_classes"], drop_rate=cfg["drop_rate"]
    )
    m.load_state_dict(ckpt["model_state"])
    m = m.to(device).eval()
    print(f"✅ EfficientNet loaded | AUC {ckpt['val_auc']:.4f}")
    return m

print("Loading models...")
effnet_model = load_effnet(EFFNET_CKPT)
yolo_model   = YOLO(YOLO_CKPT)
print("✅ YOLOv8 loaded")

Loading models...
✅ EfficientNet loaded | AUC 0.9134
✅ YOLOv8 loaded


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, "activations", o)
        )
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, "gradients", go[0].detach())
        )

    def generate(self, input_tensor):
        self.model.eval()
        out = self.model(input_tensor)
        cls = out.argmax(dim=1).item()
        self.model.zero_grad()
        out[0, cls].backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = (weights * self.activations).sum(dim=1, keepdim=True)
        cam     = torch.relu(cam).squeeze().detach().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        probs   = torch.softmax(out, dim=1)[0].detach().cpu().numpy()
        return cam, probs

def get_target_layer(model):
    try:    return model.blocks[-1][-1].conv_pwl
    except: return model.blocks[-1]


In [ ]:
def run_effnet(image_clahe):
    tensor = infer_transform(image_clahe).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(effnet_model(tensor), dim=1)[0].cpu().numpy()
    return float(probs[1])

def run_yolo(img_path):
    results    = yolo_model.predict(
        source  = img_path,
        conf    = ENSEMBLE_CONFIG["yolo_conf"],
        iou     = ENSEMBLE_CONFIG["yolo_iou"],
        imgsz   = ENSEMBLE_CONFIG["yolo_imgsz"],
        verbose = False,
    )
    result     = results[0]
    detections = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
            cf           = float(box.conf[0].cpu())
            detections.append({
                "confidence": round(cf, 3),
                "bbox"      : [int(x1), int(y1), int(x2), int(y2)],
            })
    yolo_score = max((d["confidence"] for d in detections), default=0.0)
    return yolo_score, detections

def ensemble_decision(effnet_score, yolo_score):
    w_e   = ENSEMBLE_CONFIG["effnet_weight"]
    w_y   = ENSEMBLE_CONFIG["yolo_weight"]
    score = w_e * effnet_score + w_y * yolo_score

    effnet_pred  = effnet_score >= ENSEMBLE_CONFIG["effnet_threshold"]
    yolo_pred    = yolo_score   >= ENSEMBLE_CONFIG["yolo_conf"]
    models_agree = effnet_pred  == yolo_pred
    uncertain    = not models_agree or 0.40 <= score <= 0.65
    fracture     = score >= ENSEMBLE_CONFIG["final_threshold"]

    return score, fracture, uncertain, models_agree

In [ ]:
def predict_xray(image):
    """
    Input : PIL image from Gradio
    Output: 4 images (panels) + text report
    """
    if image is None:
        return None, None, None, None, "Please upload an X-ray image."

    # ── Preprocess ───────────────────────────────────────────
    pil_orig  = Image.fromarray(image).convert("L")
    pil_clahe = apply_clahe(pil_orig)

    # Save temp file for YOLO (needs file path)
    tmp_path = "/tmp/xray_input.jpg"
    pil_orig.save(tmp_path)

    # ── Run models ───────────────────────────────────────────
    effnet_score          = run_effnet(pil_clahe)
    yolo_score, dets      = run_yolo(tmp_path)
    score, fracture, uncertain, agree = ensemble_decision(effnet_score, yolo_score)

    # ── Panel 1: CLAHE image ─────────────────────────────────
    clahe_np  = np.array(pil_clahe.resize((512, 512)))
    fig1, ax1 = plt.subplots(figsize=(5, 5))
    ax1.imshow(clahe_np, cmap="gray")
    ax1.set_title("CLAHE Enhanced X-ray", fontsize=12, fontweight="bold")
    ax1.axis("off")
    plt.tight_layout()
    panel1 = fig_to_array(fig1)
    plt.close(fig1)

    # ── Panel 2: Grad-CAM ────────────────────────────────────
    target_layer = get_target_layer(effnet_model)
    gc           = GradCAM(effnet_model, target_layer)
    input_tensor = infer_transform(pil_clahe).unsqueeze(0).to(device)
    cam, _       = gc.generate(input_tensor)

    sz          = (512, 512)
    cam_resized = cv2.resize(cam, sz)
    heatmap     = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    orig_np     = np.array(pil_clahe.resize(sz))
    orig_bgr    = cv2.cvtColor(orig_np, cv2.COLOR_GRAY2BGR)
    overlay     = cv2.addWeighted(orig_bgr, 0.55, heatmap, 0.45, 0)

    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 5))
    axes2[0].imshow(cam_resized, cmap="jet")
    axes2[0].set_title("Grad-CAM Heatmap", fontsize=11, fontweight="bold")
    axes2[0].axis("off")
    axes2[1].imshow(overlay[..., ::-1])
    axes2[1].set_title(f"Overlay | EfficientNet: {effnet_score:.3f}", fontsize=11)
    axes2[1].axis("off")
    plt.tight_layout()
    panel2 = fig_to_array(fig2)
    plt.close(fig2)

    # ── Panel 3: YOLO detections ─────────────────────────────
    orig_for_yolo = np.array(Image.open(tmp_path).convert("RGB"))
    h_y, w_y      = orig_for_yolo.shape[:2]

    fig3, ax3 = plt.subplots(figsize=(5, 5))
    ax3.imshow(orig_for_yolo)
    for det in dets:
        x1,y1,x2,y2 = det["bbox"]
        rect = patches.Rectangle(
            (x1,y1), x2-x1, y2-y1,
            linewidth=3, edgecolor="red", facecolor="none"
        )
        ax3.add_patch(rect)
        ax3.text(
            x1, max(y1-10, 0),
            f"fractured {det['confidence']:.2f}",
            color="white", fontsize=10, fontweight="bold",
            bbox=dict(facecolor="red", alpha=0.8, edgecolor="none", pad=2)
        )
    n_det = len(dets)
    title = f"YOLO: {n_det} region(s) | Score: {yolo_score:.3f}"
    ax3.set_title(title, fontsize=11, fontweight="bold",
                  color="red" if n_det > 0 else "green")
    ax3.axis("off")
    plt.tight_layout()
    panel3 = fig_to_array(fig3)
    plt.close(fig3)

    # ── Panel 4: Ensemble result ──────────────────────────────
    if uncertain:
        verdict     = "⚠️  UNCERTAIN"
        verdict_sub = "Radiologist review required"
        bar_color   = "#FFA500"
    elif fracture:
        verdict     = "🔴  FRACTURE DETECTED"
        verdict_sub = "Abnormal X-ray"
        bar_color   = "#FF4444"
    else:
        verdict     = "🟢  NORMAL"
        verdict_sub = "No fracture detected"
        bar_color   = "#44BB44"

    fig4, ax4 = plt.subplots(figsize=(6, 5))
    ax4.axis("off")

    # Score bars
    metrics = [
        ("EfficientNet", effnet_score, "#4488FF"),
        ("YOLO",         yolo_score,   "#FF8844"),
        ("Ensemble",     score,        bar_color),
    ]
    y_pos = [0.75, 0.55, 0.30]
    for (label, val, color), y in zip(metrics, y_pos):
        ax4.barh(y, val, height=0.12, color=color, alpha=0.85,
                 transform=ax4.transData)
        ax4.text(-0.02, y, label, ha="right", va="center",
                 fontsize=11, transform=ax4.transData)
        ax4.text(val + 0.02, y, f"{val:.3f}", ha="left", va="center",
                 fontsize=11, fontweight="bold", transform=ax4.transData)

    ax4.set_xlim(-0.3, 1.2)
    ax4.set_ylim(0.1, 1.0)
    ax4.axvline(x=ENSEMBLE_CONFIG["final_threshold"], color="gray",
                linestyle="--", alpha=0.5, linewidth=1.5)
    ax4.text(ENSEMBLE_CONFIG["final_threshold"], 0.15,
             f"threshold={ENSEMBLE_CONFIG['final_threshold']}",
             ha="center", fontsize=8, color="gray")

    ax4.text(0.5, 0.95, verdict, ha="center", va="top",
             fontsize=16, fontweight="bold", color=bar_color,
             transform=ax4.transAxes)
    ax4.text(0.5, 0.88, verdict_sub, ha="center", va="top",
             fontsize=11, color="gray", transform=ax4.transAxes)
    ax4.text(0.5, 0.10,
             f"Models agree: {'Yes ✓' if agree else 'No ✗'}",
             ha="center", fontsize=10, color="gray",
             transform=ax4.transAxes)

    fig4.patch.set_facecolor("#F8F8F8")
    plt.tight_layout()
    panel4 = fig_to_array(fig4)
    plt.close(fig4)

    # ── Text report ───────────────────────────────────────────
    report = f"""
ENSEMBLE FRACTURE DETECTION REPORT
{'='*40}
Final Decision  : {verdict}
Ensemble Score  : {score:.4f}

Individual Scores:
  EfficientNet  : {effnet_score:.4f}
  YOLO          : {yolo_score:.4f}

YOLO Detections : {len(dets)} region(s)
Models Agree    : {'Yes' if agree else 'No'}
Uncertain Flag  : {'Yes — review recommended' if uncertain else 'No'}

Thresholds Used:
  Ensemble      : {ENSEMBLE_CONFIG['final_threshold']}
  EfficientNet  : {ENSEMBLE_CONFIG['effnet_threshold']}
  YOLO conf     : {ENSEMBLE_CONFIG['yolo_conf']}
"""
    return panel1, panel2, panel3, panel4, report.strip()


def fig_to_array(fig):
    """Convert matplotlib figure to numpy array for Gradio."""
    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    arr = np.asarray(buf)
    return arr[..., :3]   # drop alpha


In [ ]:
with gr.Blocks(
    title="Bone Fracture Detection",
    theme=gr.themes.Soft(),
    css="""
        .header { text-align: center; margin-bottom: 20px; }
        .result-box { border-radius: 10px; padding: 10px; }
        footer { display: none !important; }
    """
) as demo:

    gr.HTML("""
        <div class="header">
            <h1>🦴 Bone Fracture Detection System</h1>
            <p style="color: gray; font-size: 14px;">
                EfficientNet-B2 (AUC 0.913) + YOLOv8l (mAP@0.5 0.922) Ensemble
            </p>
            <p style="color: #cc4444; font-size: 12px;">
                ⚠️ For research purposes only. Not a substitute for professional medical diagnosis.
            </p>
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            inp_image = gr.Image(
                label="Upload X-ray Image",
                type="numpy",
                height=350,
            )
            run_btn = gr.Button(
                "🔍 Analyze X-ray",
                variant="primary",
                size="lg"
            )
            gr.Examples(
                examples=[],   # add example image paths here if available
                inputs=inp_image,
            )

        with gr.Column(scale=2):
            with gr.Row():
                out_clahe  = gr.Image(label="CLAHE Enhanced",  height=280)
                out_gradcam= gr.Image(label="Grad-CAM",        height=280)
            with gr.Row():
                out_yolo   = gr.Image(label="YOLO Detection",  height=280)
                out_result = gr.Image(label="Ensemble Scores", height=280)

    with gr.Row():
        out_report = gr.Textbox(
            label="Detailed Report",
            lines=15,
            max_lines=20,
        )

    gr.HTML("""
        <div style="text-align:center; color:gray; font-size:12px; margin-top:10px;">
            <b>How to read results:</b>
            CLAHE = contrast-enhanced input |
            Grad-CAM = what EfficientNet focuses on |
            YOLO = fracture localization boxes |
            Ensemble = combined decision
        </div>
    """)

    run_btn.click(
        fn      = predict_xray,
        inputs  = [inp_image],
        outputs = [out_clahe, out_gradcam, out_yolo, out_result, out_report],
    )

    # Also trigger on image upload
    inp_image.change(
        fn      = predict_xray,
        inputs  = [inp_image],
        outputs = [out_clahe, out_gradcam, out_yolo, out_result, out_report],
    )


In [ ]:
demo.launch(
    share          = True,    # generates public URL — works from Colab
    debug          = False,
    show_error     = True,
    server_port    = 7860,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f693dce9ac967a8c62.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
